# Chunking de manifestações longas
**Fernando Amorim Pontes Neto**

Comparamos (240, 0), (240, 60) e (400, 80), em caracteres. As duas primeiras configurações isolam o efeito do overlap. A terceira explora mais contexto por chunk. A base do enunciado descreve cinco textos com mais de 500 caracteres; o contexto menciona palavras, mas a especificação do dataset usa caracteres.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

In [ ]:
longas=base.assign(tamanho=base.texto.str.len()).sort_values(['tamanho','id'],ascending=[False,True]).head(5)
display(longas[['id','tamanho','texto']])
assert (longas.tamanho>500).all(), 'As cinco maiores manifestações devem ter mais de 500 caracteres.'
configuracoes=[(240,0),(240,60),(400,80)]
resumos=[]; detalhes=[]
for tamanho,overlap in configuracoes:
    registros=[]
    for r in longas.itertuples():
        for chunk in dividir(r.texto,tamanho,overlap):
            registros.append({'id':r.id,**chunk})
    chunks=pd.DataFrame(registros)
    e=codificar(chunks.texto.tolist()); sim=e@e.T
    adj=[]; sobreposicoes=[]; intra=[]; inter=[]
    for i in range(len(chunks)):
        for j in range(i+1,len(chunks)):
            (intra if chunks.id.iloc[i]==chunks.id.iloc[j] else inter).append(float(sim[i,j]))
        if i+1<len(chunks) and chunks.id.iloc[i]==chunks.id.iloc[i+1]:
            adj.append(float(sim[i,i+1]))
            sobreposicoes.append(max(0,int(chunks.inicio.iloc[i]+chunks.caracteres.iloc[i]-chunks.inicio.iloc[i+1])))
    r={'chunk_size':tamanho,'chunk_overlap':overlap,'n_chunks':len(chunks),
       'cosseno_adjacente':float(np.mean(adj)) if adj else None,
       'overlap_efetivo_medio':float(np.mean(sobreposicoes)) if sobreposicoes else 0,
       'cosseno_mesma_manifestacao':float(np.mean(intra)) if intra else None,
       'cosseno_outras_manifestacoes':float(np.mean(inter)) if inter else None}
    resumos.append(r)
    chunks.to_csv(saida/f'chunks_{tamanho}_{overlap}.csv',index=False)
    display(Markdown(f'### Configuração {tamanho} e {overlap}'))
    display(chunks)
    coords=projetar(e,'PCA'); fig,ax=plt.subplots(figsize=(10,5))
    for ident in longas.id:
        mask=(chunks.id==ident).to_numpy()
        ax.scatter(coords[mask,0],coords[mask,1],label=ident)
    for i,xy in enumerate(coords): ax.annotate(f'{chunks.id.iloc[i]} C{chunks.chunk.iloc[i]}',xy,fontsize=7)
    ax.legend(); ax.set_title(f'PCA dos chunks — tamanho {tamanho}, overlap {overlap}')
    fig.tight_layout(); fig.savefig(saida/f'chunks_pca_{tamanho}_{overlap}.png',dpi=150); plt.show()
    exemplo=chunks[chunks.id==longas.id.iloc[0]].head(2)
    detalhes.append({'configuracao':[tamanho,overlap],'exemplo':exemplo.to_dict('records')})
display(pd.DataFrame(resumos).round(4))
(saida/'chunking_resumo.json').write_text(json.dumps({'configuracoes':resumos,'exemplos':detalhes},ensure_ascii=False,indent=2))

In [ ]:
a,b,c=resumos
mudanca=b['cosseno_adjacente']-a['cosseno_adjacente']
display(Markdown(f"Mantendo chunk_size=240, acrescentar overlap=60 mudou o cosseno médio dos vizinhos em **{mudanca:+.3f}**. Parte da variação pode vir das palavras repetidas e não de melhor compreensão. O overlap efetivo médio foi **{b['overlap_efetivo_medio']:.1f} caracteres**, pois o separador recursivo respeita os limites disponíveis."))
for r in resumos:
    intra=r['cosseno_mesma_manifestacao']; inter=r['cosseno_outras_manifestacoes']
    display(Markdown(f"Em ({r['chunk_size']}, {r['chunk_overlap']}), a média de cosseno entre chunks da mesma manifestação é **{intra:.3f}**, contra **{inter:.3f}** entre manifestações. "+('Há proximidade relativa no espaço original.' if intra>inter else 'A origem comum não produziu maior proximidade média.')))
for detalhe in detalhes:
    display(Markdown(f"**Exemplo de limites — {detalhe['configuracao']}**"))
    for trecho in detalhe['exemplo']:
        print(f"{trecho['id']} / chunk {trecho['chunk']} / início {trecho['inicio']}: {trecho['texto']}")

## Interpretação e escolha
A configuração (400, 80) é a escolha inicial para a interface quando a denúncia descreve local, ocorrência e consequência em frases consecutivas: o trecho maior oferece mais espaço para essas informações permanecerem juntas. Os exemplos acima permitem conferir os limites reais; um score alto não basta para concluir que o sentido foi preservado. (240, 60) favorece recuperação de detalhes e custa mais vetores; (240, 0) não repete contexto e pode separar uma referência do objeto a que se refere. O overlap deve ser escolhido após inspecionar os trechos e avaliar consultas representativas. A PCA é apenas diagnóstico visual; as médias no espaço original são a evidência quantitativa de proximidade. Janelas semânticas podem ainda separar temas diferentes dentro de uma mesma denúncia.

## Referências
Material fornecido: app_chunks_embeddings.py. RecursiveCharacterTextSplitter: https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter. PCA e t-SNE: https://scikit-learn.org/stable/modules/manifold.html.